<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/%D0%BA%D1%812.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [63]:
import pandas as pd
from openpyxl import load_workbook

def parse_ks2(file_path):
    # Загружаем файл
    wb = load_workbook(file_path)
    ws = wb['КС-2 ИТОГО']

    # Определяем стартовую строку (после заголовков)
    start_row = 25
    data = []

    # Проходим по всем строкам листа
    for row_num, row in enumerate(ws.iter_rows(min_row=start_row, values_only=True), start=start_row):
        # Получаем значения из столбцов C-N (индексы 2-13)
        row_data = list(row[2:14])

        # Проверяем, есть ли данные в строке
        if any(cell is not None and cell != '' for cell in row_data):
            # Добавляем номер строки для отслеживания
            row_data.append(row_num)
            data.append(row_data)

    # Создаем DataFrame с правильными названиями столбцов
    columns = [
        'п/п',                              # Столбец C (индекс 2)
        'Подпозиция',                       # Столбец D (индекс 3) - безымянный столбец
        'поз. по сме-те',                   # Столбец E (индекс 4)
        'Шифр расценки и коды ресурсов',    # Столбец F (индекс 5)
        'Наименование работ и затрат',      # Столбец G (индекс 6)
        'Единица измерения',                # Столбец H (индекс 7)
        'Кол-во единиц',                    # Столбец I (индекс 8)
        'Цена на единицу измерения, руб.',  # Столбец J (индекс 9)
        'Попра-вочные коэфф., нормы НР и СП', # Столбец K (индекс 10)
        'Всего затрат в базисном уровне цен, руб.', # Столбец L (индекс 11)
        'Индексы пересчета, нормы НР и СП', # Столбец M (индекс 12)
        'ВСЕГО затрат, руб.',               # Столбец N (индекс 13)
        'Номер строки в Excel'              # Дополнительный столбец для отладки
    ]

    df = pd.DataFrame(data, columns=columns)

    # Очищаем данные от служебных символов и лишних пробелов
    for col in df.columns:
        df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) and x is not None else x)

    # Заменяем значения, которые могут быть формулами или ошибками
    df = df.replace({'=IF(D35="",0,2)': None, '=K23': None}, regex=True)

    return df

# Использование
df = parse_ks2('КС-2.xlsx')

In [64]:
df

,п/п,Подпозиция,поз. по сме-те,Шифр расценки и коды ресурсов,Наименование работ и затрат,Единица измерения,Кол-во единиц,"Цена на единицу измерения, руб.","Попра-вочные коэфф., нормы НР и СП","Всего затрат в базисном уровне цен, руб.","Индексы пересчета, нормы НР и СП","ВСЕГО затрат, руб.",Номер строки в Excel
0,None,ИТОГОВЫЙ AKT,None,None,None,None,None,None,None,None,None,None,25
1,None,О ПРИЕМКЕ ВЫПОЛНЕННЫХ РАБОТ,None,None,None,None,None,None,None,None,None,None,26
2,Номер,None,None,Шифр расценки и коды ресурсов,Наименование работ и затрат,Единица измерения,Кол-во единиц,"Цена на единицу измерения, руб.","Попра-вочные коэфф., нормы НР и СП","Всего затрат в базисном уровне цен, руб.","Индексы пересчета, нормы НР и СП","ВСЕГО затрат, руб.",28
3,п/п,None,поз. по сме-те,None,None,None,None,None,None,None,None,None,29
4,1,None,2,3,4,5,6,7,8,9,10,11,30
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16551,None,None,None,None,None,None,None,None,None,"ВСЕГО ПО АКТУ формы КС-2, руб.:",=SUM(P35:P17717),None,17720
16552,Сдал,None,None,Генеральный директор \nОбщество с ограниченной...,None,None,None,None,None,Д.В. Петухов,None,None,17724
16553,None,None,None,None,None,"[должность,подпись(инициалы,фамилия)]",None,None,None,None,None,None,17725
16554,Принял,None,None,Генеральный директор \nОбщество с ограниченной...,None,None,None,None,None,А.А. Лебедева,None,None,17728


In [65]:
# Удаляем строки с индексами от 0 до 7 включительно
df = df.iloc[8:]

In [66]:
df

,п/п,Подпозиция,поз. по сме-те,Шифр расценки и коды ресурсов,Наименование работ и затрат,Единица измерения,Кол-во единиц,"Цена на единицу измерения, руб.","Попра-вочные коэфф., нормы НР и СП","Всего затрат в базисном уровне цен, руб.","Индексы пересчета, нормы НР и СП","ВСЕГО затрат, руб.",Номер строки в Excel
8,1,1,1,09-08-001-03,Установка металлических столбов высотой до 4 м...,100 ШТ,0.79,None,None,None,,None,36
9,None,None,None,None,"Объем: 0,79=79/100",None,None,None,None,None,None,None,37
10,None,None,None,None,ЗП,None,None,375.75,,296.84,7.56,2244.13,38
11,None,None,None,None,ЭМ,None,None,732.4,,578.6,7.56,4374.19,39
12,None,None,None,None,в т.ч. ЗПМ,None,None,345.45,,272.91,7.56,2063.17,40
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16551,None,None,None,None,None,None,None,None,None,"ВСЕГО ПО АКТУ формы КС-2, руб.:",=SUM(P35:P17717),None,17720
16552,Сдал,None,None,Генеральный директор \nОбщество с ограниченной...,None,None,None,None,None,Д.В. Петухов,None,None,17724
16553,None,None,None,None,None,"[должность,подпись(инициалы,фамилия)]",None,None,None,None,None,None,17725
16554,Принял,None,None,Генеральный директор \nОбщество с ограниченной...,None,None,None,None,None,А.А. Лебедева,None,None,17728


In [67]:
# Количество уникальных значений в колонке "Шифр расценки и коды ресурсов"
unique_codes_count = df["Шифр расценки и коды ресурсов"].nunique()
print(f"Уникальных значений в колонке 'Шифр расценки и коды ресурсов': {unique_codes_count}")

Уникальных значений в колонке 'Шифр расценки и коды ресурсов': 1939


In [68]:
# Сначала создадим копию DataFrame, чтобы избежать возможных ошибок
df_copy = df.copy()

# Далее заменяем пустые строки ('') на NA-значения
df_copy.loc[:, 'Шифр расценки и коды ресурсов'] = df_copy['Шифр расценки и коды ресурсов'].replace('', pd.NA)

# Теперь можем убрать строки с пустыми значениями
df_cleaned = df_copy.dropna(subset=['Шифр расценки и коды ресурсов'])



In [69]:
df_cleaned

,п/п,Подпозиция,поз. по сме-те,Шифр расценки и коды ресурсов,Наименование работ и затрат,Единица измерения,Кол-во единиц,"Цена на единицу измерения, руб.","Попра-вочные коэфф., нормы НР и СП","Всего затрат в базисном уровне цен, руб.","Индексы пересчета, нормы НР и СП","ВСЕГО затрат, руб.",Номер строки в Excel
8,1,1,1,09-08-001-03,Установка металлических столбов высотой до 4 м...,100 ШТ,0.79,None,None,None,,None,36
13,None,2,"1,1",23.3.08.02-0157,Трубы стальные прямоугольные (ГОСТ 8645-86) ра...,м,180,58.08,,10454.4,7.56,79035.26,41
14,None,3,"1,2",01.7.15.02-0051,Болты анкерные,т,0.0237,10068,,238.61,7.56,1803.9,42
19,2,4,2,м38-01-003-04,Изготовление прогонов ограждения забора из ста...,т,0.723,None,None,None,,None,47
24,None,5,"2,1",23.3.08.02-0064,Трубы стальные прямоугольные (ГОСТ 8645-86) ра...,м,498,13.29,,6618.42,7.56,50035.26,52
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16547,3514,184,10,Акт № 12 от 30.10.2020,"Генподрядные услуги по приемке работ, выполняе...",компл.,1,39666.39,None,39666.39,1,39666.39,17714
16548,3515,185,11,Акт № 12 от 30.10.2020,"Генподрядные услуги по сбору, проверке, выдаче...",компл.,1,66109.52,None,66109.52,1,66109.52,17715
16549,3516,186,12,Акт № 12 от 30.10.2020,Генподрядные услуги по контролю соблюдения тех...,компл.,1,2576.53,None,2576.53,1,2576.53,17716
16552,Сдал,None,None,Генеральный директор \nОбщество с ограниченной...,None,None,None,None,None,Д.В. Петухов,None,None,17724


In [70]:
# Проверяем общее количество строк в очищенном DataFrame
total_rows_cleaned = len(df_cleaned)

# Проверяем количество пустых значений в колонке "ВСЕГО затрат, руб."
empty_total_cost_cleaned = df_cleaned['ВСЕГО затрат, руб.'].isnull().sum()

# Рассчитываем долю пустых значений
percentage_empty_cleaned = (empty_total_cost_cleaned / total_rows_cleaned) * 100

# Выводим результаты
print(f"Всего строк в очищенном DataFrame: {total_rows_cleaned}")
print(f"Пустых значений в колонке 'ВСЕГО затрат, руб.': {empty_total_cost_cleaned}")
print(f"Доля пустых значений (%): {percentage_empty_cleaned:.2f}%")

Всего строк в очищенном DataFrame: 3820
Пустых значений в колонке 'ВСЕГО затрат, руб.': 1443
Доля пустых значений (%): 37.77%


In [71]:
# Создаем полноценную копию очищенного DataFrame
df_final = df_cleaned.copy()

# Проверяем количество пустых значений в колонке "ВСЕГО затрат, руб."
empty_total_cost = df_final['ВСЕГО затрат, руб.'].isnull().sum()

# Удаляем строки с пустыми значениями в колонке "ВСЕГО затрат, руб."
df_final = df_final.dropna(subset=['ВСЕГО затрат, руб.'])

# Проверяем успешность удаления пустых значений
assert df_final['ВСЕГО затрат, руб.'].isnull().sum() == 0, \
    "В колонке 'ВСЕГО затрат, руб.' остались пустые значения!"

# Выводим размеры DataFrame до и после очистки
initial_size = len(df_cleaned)
final_size = len(df_final)
removed_rows = initial_size - final_size
remaining_percentage = (final_size / initial_size) * 100

# Выводим результаты
print(f"Текущий размер очищенного DataFrame: {initial_size} строк")
print(f"Удалено строк с пустыми значениями в колонке 'ВСЕГО затрат, руб.': {removed_rows}")
print(f"Итоговый размер DataFrame после очистки: {final_size} строк")
print(f"Доля оставшихся строк (%): {remaining_percentage:.2f}%")

Текущий размер очищенного DataFrame: 3820 строк
Удалено строк с пустыми значениями в колонке 'ВСЕГО затрат, руб.': 1443
Итоговый размер DataFrame после очистки: 2377 строк
Доля оставшихся строк (%): 62.23%


In [72]:
df_final

,п/п,Подпозиция,поз. по сме-те,Шифр расценки и коды ресурсов,Наименование работ и затрат,Единица измерения,Кол-во единиц,"Цена на единицу измерения, руб.","Попра-вочные коэфф., нормы НР и СП","Всего затрат в базисном уровне цен, руб.","Индексы пересчета, нормы НР и СП","ВСЕГО затрат, руб.",Номер строки в Excel
13,None,2,"1,1",23.3.08.02-0157,Трубы стальные прямоугольные (ГОСТ 8645-86) ра...,м,180,58.08,,10454.4,7.56,79035.26,41
14,None,3,"1,2",01.7.15.02-0051,Болты анкерные,т,0.0237,10068,,238.61,7.56,1803.9,42
24,None,5,"2,1",23.3.08.02-0064,Трубы стальные прямоугольные (ГОСТ 8645-86) ра...,м,498,13.29,,6618.42,7.56,50035.26,52
34,None,7,"3,2",07.2.07.13,Конструкции стальные,т,0.723,0,,0,7.56,0,62
45,None,9,"4,1",08.3.09.01-0008,"Профилированный лист оцинкованный Н60-845-0,55",т,1.523149,11817.6,,17999.97,7.56,136079.74,73
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16545,3512,182,8,Акт № 12 от 30.10.2020,Генподрядные услуги по сбору объемов работ на ...,компл.,1,52889.37,None,52889.37,1,52889.37,17712
16546,3513,183,9,Акт № 12 от 30.10.2020,"Генподрядные услуги по проверке, выдаче замеча...",компл.,1,39666.39,None,39666.39,1,39666.39,17713
16547,3514,184,10,Акт № 12 от 30.10.2020,"Генподрядные услуги по приемке работ, выполняе...",компл.,1,39666.39,None,39666.39,1,39666.39,17714
16548,3515,185,11,Акт № 12 от 30.10.2020,"Генподрядные услуги по сбору, проверке, выдаче...",компл.,1,66109.52,None,66109.52,1,66109.52,17715


In [110]:
# Регулярное выражение для нахождения строк с любыми буквами
pattern = r'[A-Za-zА-Яа-яЁё]'

# Оставляем только строки, содержащие буквы
df_final = df_final[
    df_final['Шифр расценки и коды ресурсов'].astype(str).str.contains(pattern)
]


In [112]:
# Подсчет количества уникальных значений в колонке "Шифр расценки и коды ресурсов"
unique_codes = df_final['Шифр расценки и коды ресурсов'].nunique()

# Получение списка уникальных значений
unique_code_list = df_final['Шифр расценки и коды ресурсов'].unique()

# Расчёт распределения значений
code_distribution = df_final['Шифр расценки и коды ресурсов'].value_counts(normalize=True) * 100

# Вывод результатов
print(f"Количество уникальных значений в колонке 'Шифр расценки и коды ресурсов': {unique_codes}\n")
print("Список уникальных значений:")
for code in unique_code_list:
    print(code)

Количество уникальных значений в колонке 'Шифр расценки и коды ресурсов': 1017

Список уникальных значений:
Счет-фактура № 675 от 3 сентября 2019г
Счет-фактура № 660 от 28.08.2019г
т01-01-01-016
т01-01-01-043
Акт № 1 от 10.03.2020
т01-01-01-039
Акт № 1 от 30.04.2020
Счет-фактура № ФМ9532 от 26 марта 2020 г.
Акт №02 от 29.05.2020
Акт № 3 от 29.05.2020
Ц.поставщ.
Ц.пост.
Сч.факт. № КА-1350 от 22.05.2020г
т01-01-01-003
т01-01-01-042
т01-01-01-014
т01-01-01-043методические рекомендации Минстроя России №519/ пр от 04 09 2019
Сч.факт.№ШД000272513/01от 10 июня 2020г
Сч.фактура № 850от 19.06.2020г
ССЦ-11.2.08.02-0010
ССЦ-01.6.01.10-0015
ССЦ-12.1.02.10-0095
ССЦ-01.2.01.02-0052
ССЦ-14.3.01.01-0001
ССЦ-01.7.06.09-0007
ФССЦ-19.1.01.03-0057
ФССЦ-19.1.01.03-0033
ФССЦ-19.1.01.03-0024
ФССЦ-19.1.01.03-0022
ФССЦ-19.1.01.03-0021
ФССЦ-19.1.01.03-0047
ФССЦ-19.1.01.03-0046
ФССЦ-19.1.01.03-0045
ФССЦ-19.1.01.03-0044
ФССЦ-19.1.01.03-0071
Сч.фак. №584 от18 июня 2020г
ФССЦ-19.3.01.01-0013
ФССЦ-19.3.01.01-0012
ФС

In [105]:
# Подсчет количества уникальных значений в колонке "Единица измерения"
unique_units = df_final['Единица измерения'].nunique()

# Получение списка уникальных значений
unique_unit_list = df_final['Единица измерения'].unique()

# Расчёт распределения значений
unit_distribution = df_final['Единица измерения'].value_counts(normalize=True) * 100

# Вывод результатов
print(f"Количество уникальных значений в колонке 'Единица измерения': {unique_units}\n")

print("\nРаспределение значений (проценты):")
print(unit_distribution.round(2))

Количество уникальных значений в колонке 'Единица измерения': 37


Распределение значений (проценты):
Единица измерения
шт.          26.08
т            13.21
ШТ           11.70
м             9.97
м3            8.58
м2            5.81
компл.        5.43
10 ШТ         4.33
кг            2.02
10 шт.        1.94
1000 м        1.77
1 Т ГРУЗА     1.64
10 м          1.51
100 шт.       1.22
л             0.97
1000 м2       0.84
100 ШТ        0.63
1000 шт.      0.34
К-Т           0.21
100 м         0.21
км            0.21
м.п.          0.21
П.М           0.17
шт..          0.13
              0.13
услуга        0.08
шт            0.08
лист          0.08
1000М         0.08
упак          0.08
кг.           0.08
КОМПЛ         0.04
набор         0.04
1000 ШТ       0.04
10 м2         0.04
100 м2        0.04
сут.          0.04
Name: proportion, dtype: float64


In [113]:
# Регулярное выражение для точного поиска слова "акт"
pattern = r'\bакт\b'

# Фильтруем строки с указанным словом
df_akt = df_final[
    df_final['Шифр расценки и коды ресурсов'].str.contains(pattern, regex=True, case=False)
]

# Сохраняем результат в Excel
output_file_path = 'filtered_exact_acts.xlsx'
df_akt.to_excel(output_file_path, index=False)

print(f'Файл успешно сохранён: {output_file_path}')

Файл успешно сохранён: filtered_exact_acts.xlsx


In [114]:
df_akt

,п/п,Подпозиция,поз. по сме-те,Шифр расценки и коды ресурсов,Наименование работ и затрат,Единица измерения,Кол-во единиц,"Цена на единицу измерения, руб.","Попра-вочные коэфф., нормы НР и СП","Всего затрат в базисном уровне цен, руб.","Индексы пересчета, нормы НР и СП","ВСЕГО затрат, руб.",Номер строки в Excel
131,16,3,1,Акт № 1 от 10.03.2020,Подбор и изучение юридической и технической до...,компл.,1,311765,None,=J189*I189,1,=L189*M189,189
132,17,4,2,Акт № 1 от 10.03.2020,"Установление наличия, сбор и оценка состояния ...",компл.,1,388235,None,=J190*I190,1,=L190*M190,190
133,18,5,3,Акт № 1 от 10.03.2020,Получение сведений о наличии особо охраняемой ...,компл.,1,317647.0,None,=J191*I191,1,=L191*M191,191
134,19,6,4,Акт № 1 от 10.03.2020,Получение сведений о наличии древесины и ценны...,компл.,1,300000,None,=J192*I192,1,=L192*M192,192
135,20,7,5,Акт № 1 от 10.03.2020,Обозначение на местности вырубаемых деревьев: ...,компл.,1,294118,None,=J193*I193,1,=L193*M193,193
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16545,3512,182,8,Акт № 12 от 30.10.2020,Генподрядные услуги по сбору объемов работ на ...,компл.,1,52889.37,None,52889.37,1,52889.37,17712
16546,3513,183,9,Акт № 12 от 30.10.2020,"Генподрядные услуги по проверке, выдаче замеча...",компл.,1,39666.39,None,39666.39,1,39666.39,17713
16547,3514,184,10,Акт № 12 от 30.10.2020,"Генподрядные услуги по приемке работ, выполняе...",компл.,1,39666.39,None,39666.39,1,39666.39,17714
16548,3515,185,11,Акт № 12 от 30.10.2020,"Генподрядные услуги по сбору, проверке, выдаче...",компл.,1,66109.52,None,66109.52,1,66109.52,17715


In [125]:
# Оставляем только строки, НЕ содержащие слово "акт"
df_filtered_final = df_final[
    ~df_final['Шифр расценки и коды ресурсов'].str.contains(pattern, regex=True, case=False)
]


In [126]:
df_filtered_final

,п/п,Подпозиция,поз. по сме-те,Шифр расценки и коды ресурсов,Наименование работ и затрат,Единица измерения,Кол-во единиц,"Цена на единицу измерения, руб.","Попра-вочные коэфф., нормы НР и СП","Всего затрат в базисном уровне цен, руб.","Индексы пересчета, нормы НР и СП","ВСЕГО затрат, руб.",Номер строки в Excel
64,None,12,"6,1",Счет-фактура № 675 от 3 сентября 2019г,Модульное здание из 2-х блок- контейнеров разм...,шт.,1,34391.53,,34391.53,7.56,259999.97,96
65,None,13,"6,2",Счет-фактура № 660 от 28.08.2019г,Блок- контейнер (улучшенный) разм.5850х2400х2...,шт.,3,9589.95,,28769.85,7.56,217500.07,97
85,8,2,2,т01-01-01-016,Погрузочные работы при автомобильных перевозка...,1 Т ГРУЗА,2.75,10.45,,28.74,7.56,217.26,123
101,11,5,5,т01-01-01-043,Погрузочные работы при автомобильных перевозка...,1 Т ГРУЗА,696.57,3.28,,2284.75,7.56,17272.71,143
121,14,2,2,т01-01-01-043,Погрузочные работы при автомобильных перевозка...,1 Т ГРУЗА,607.17,3.28,,1991.52,7.56,15055.87,171
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16485,3471,=D17632+1,3,договорная,Пандус в бокс 230х1200х1800 мм. Металлокаркас ...,компл.,2,39683.336,None,=J17633*I17633,1,=L17633*M17633,17633
16504,3477,14,1,договорная,Сопровождение строительно-монтажных работ со с...,компл.,1,450000,None,450000,1,450000,17665
16505,3478,15,2,договорная,Разработка паспорта благоустройства,компл.,1,400000,None,400000,1,400000,17666
16506,3479,16,3,договорная,Внесение сведений в АИС «Реестр зеленых насажд...,компл.,1,250000,None,250000,1,250000,17667


In [138]:
# Регулярное выражение для удаления строк, начинающихся с "ФССЦ"
fss_pattern = r'^ФССЦ.*'
ss_pattern = r'^ССЦ.*'
fer_pattern = r'^ФЕР.*'
fer_pattern = r'^ФЕР.*'
t_pattern = r'^т01.*'
fs_pattern = r'^ФСЦ.*'

In [139]:
mask_fss = ~df_filtered_final['Шифр расценки и коды ресурсов'].str.contains(fss_pattern, regex=True, case=False)
mask_fer = ~df_filtered_final['Шифр расценки и коды ресурсов'].str.contains(fer_pattern, regex=True, case=False)
mask_ss = ~df_filtered_final['Шифр расценки и коды ресурсов'].str.contains(ss_pattern, regex=True, case=False)
mask_ss = ~df_filtered_final['Шифр расценки и коды ресурсов'].str.contains(t_pattern, regex=True, case=False)
mask_ss = ~df_filtered_final['Шифр расценки и коды ресурсов'].str.contains(fs_pattern, regex=True, case=False)


# Логическое "И": Все маски должны быть истинны
final_mask = mask_fss & mask_fer & mask_ss

df_filtered_final = df_filtered_final[final_mask]

In [140]:
df_filtered_final

,п/п,Подпозиция,поз. по сме-те,Шифр расценки и коды ресурсов,Наименование работ и затрат,Единица измерения,Кол-во единиц,"Цена на единицу измерения, руб.","Попра-вочные коэфф., нормы НР и СП","Всего затрат в базисном уровне цен, руб.","Индексы пересчета, нормы НР и СП","ВСЕГО затрат, руб.",Номер строки в Excel
64,None,12,"6,1",Счет-фактура № 675 от 3 сентября 2019г,Модульное здание из 2-х блок- контейнеров разм...,шт.,1,34391.53,,34391.53,7.56,259999.97,96
65,None,13,"6,2",Счет-фактура № 660 от 28.08.2019г,Блок- контейнер (улучшенный) разм.5850х2400х2...,шт.,3,9589.95,,28769.85,7.56,217500.07,97
959,None,49,"25,1",Счет-фактура № ФМ9532 от 26 марта 2020 г.,"Грунтовка SigmaFast 278(расход 0,225л/м2 толщ....",л,26.9,78.43,,2109.77,7.68,16203.01,1090
972,None,53,"26,2",Счет-фактура № ФМ9532 от 26 марта 2020 г.,"Эмаль SigmaDur 520(расход 0,105л/м2толщ.60мкр....",л,12.6,98.04,,1235.3,7.68,9487.13,1103
1125,None,88,"41,1",Счет-фактура № ФМ9532 от 26 марта 2020 г.,"Грунтовка SigmaFast 278(расход 0,225л/м2 толщ....",л,4.4,78.43,,345.09,7.68,2650.31,1260
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16485,3471,=D17632+1,3,договорная,Пандус в бокс 230х1200х1800 мм. Металлокаркас ...,компл.,2,39683.336,None,=J17633*I17633,1,=L17633*M17633,17633
16504,3477,14,1,договорная,Сопровождение строительно-монтажных работ со с...,компл.,1,450000,None,450000,1,450000,17665
16505,3478,15,2,договорная,Разработка паспорта благоустройства,компл.,1,400000,None,400000,1,400000,17666
16506,3479,16,3,договорная,Внесение сведений в АИС «Реестр зеленых насажд...,компл.,1,250000,None,250000,1,250000,17667


In [141]:
# Подсчет количества уникальных значений в колонке "Шифр расценки и коды ресурсов"
unique_codes = df_filtered_final['Шифр расценки и коды ресурсов'].nunique()

# Получение списка уникальных значений
unique_code_list = df_filtered_final['Шифр расценки и коды ресурсов'].unique()

# Расчёт распределения значений
code_distribution = df_filtered_final['Шифр расценки и коды ресурсов'].value_counts(normalize=True) * 100

# Вывод результатов
print(f"Количество уникальных значений в колонке 'Шифр расценки и коды ресурсов': {unique_codes}\n")
print("Список уникальных значений:")
for code in unique_code_list:
    print(code)

Количество уникальных значений в колонке 'Шифр расценки и коды ресурсов': 196

Список уникальных значений:
Счет-фактура № 675 от 3 сентября 2019г
Счет-фактура № 660 от 28.08.2019г
Счет-фактура № ФМ9532 от 26 марта 2020 г.
Ц.поставщ.
Ц.пост.
Сч.факт. № КА-1350 от 22.05.2020г
Сч.факт.№ШД000272513/01от 10 июня 2020г
Сч.фактура № 850от 19.06.2020г
Сч.фак. №584 от18 июня 2020г
Сч.факт.№848 от 18июня 2020г.
Сч.факт.№851 от 23июня 2020г.
Сч.факт.№50150от 24 июля 2020г
Сч.факт.№774от 29 июня 2020г
Сч.факт.№ 262 от 29 мая 2020г.
Сч.фактура №АН-0008898 от20 июля 2020г
Сч.фактура №АН-0008450 от13 июля 2020г
Сч.фактура №ШД000270242/01 от 25июня 2020г
Сч.фактура №04073/20 от 28 июля 2020г
Сч.факт.№ 262 от 29 мая 2020г
Сч.фактура № 852от 19.06.2020г
Сч.Фак.№ 103 от 23 июня 2020г
Сч.факт.№ 94 от 30.06.2020г
Сч.факт.№637 от16.07.2020
Сч.факт.№ 451 от 22.07.2020
Сч.факт.№466 от 12.08.2020г
Сч.факт.№451 от 22.07.2020г
Сч.факт.№209 от 31.07.2020г.2020г
Сч.факт.№ 057/17508 от 03.08.2020г
Сч.факт.№ 057/172

In [142]:
def filter_by_codes(input_codes):
    """
    Фильтрует df_filtered_final по заданным шифрам расценок и выводит сокращённый результат

    Parameters:
    input_codes (str или list): Шифр(ы) расценок для поиска
    """
    # Преобразуем вход в список, если передан одиночный код
    if isinstance(input_codes, str):
        input_codes = [input_codes]

    # Определяем интересующие нас столбцы
    columns_to_show = [
        'Шифр расценки и коды ресурсов',
        'Наименование работ и затрат',
        'Единица измерения',
        'Кол-во единиц',
        'Цена на единицу измерения, руб.'
    ]

    # Фильтруем DataFrame и выводим выбранные столбцы
    filtered_result = df_filtered_final[
        df_filtered_final['Шифр расценки и коды ресурсов'].isin(input_codes)
    ][columns_to_show]


    return filtered_result

# Пример использования:
# filter_by_codes("Акт № 1 от 10.03.2020")

In [152]:
filter_by_codes(["Счет-фактура № 675 от 3 сентября 2019г", 'Счет-фактура № ФМ9532 от 26 марта 2020 г.', 'Сч.факт. № КА-1350 от 22.05.2020г'])

,Шифр расценки и коды ресурсов,Наименование работ и затрат,Единица измерения,Кол-во единиц,"Цена на единицу измерения, руб."
64,Счет-фактура № 675 от 3 сентября 2019г,Модульное здание из 2-х блок- контейнеров разм...,шт.,1,34391.53
959,Счет-фактура № ФМ9532 от 26 марта 2020 г.,"Грунтовка SigmaFast 278(расход 0,225л/м2 толщ....",л,26.9,78.43
972,Счет-фактура № ФМ9532 от 26 марта 2020 г.,"Эмаль SigmaDur 520(расход 0,105л/м2толщ.60мкр....",л,12.6,98.04
1125,Счет-фактура № ФМ9532 от 26 марта 2020 г.,"Грунтовка SigmaFast 278(расход 0,225л/м2 толщ....",л,4.4,78.43
1138,Счет-фактура № ФМ9532 от 26 марта 2020 г.,"Эмаль SigmaDur 520(расход 0,105л/м2толщ.60мкр....",л,2.1,98.04
1516,Сч.факт. № КА-1350 от 22.05.2020г,"Гетекстиль Нетканный ""Геофлакс"" 200 2*1000м Це...",м2,2155,3.03


In [149]:
# Список значений для удаления
values_to_remove = ["Ц.поставщ.", "Ц.пост.", 'Счет-фактура № ФМ9532 от 26 марта 2020 г.', 'Сч.факт. № КА-1350 от 22.05.2020г']

# Фильтруем DataFrame, оставляя только строки, не входящие в список значений
filtered_df = df_filtered_final[
    ~df_filtered_final['Шифр расценки и коды ресурсов'].isin(values_to_remove)
]


In [150]:
filtered_df

,п/п,Подпозиция,поз. по сме-те,Шифр расценки и коды ресурсов,Наименование работ и затрат,Единица измерения,Кол-во единиц,"Цена на единицу измерения, руб.","Попра-вочные коэфф., нормы НР и СП","Всего затрат в базисном уровне цен, руб.","Индексы пересчета, нормы НР и СП","ВСЕГО затрат, руб.",Номер строки в Excel
64,None,12,"6,1",Счет-фактура № 675 от 3 сентября 2019г,Модульное здание из 2-х блок- контейнеров разм...,шт.,1,34391.53,,34391.53,7.56,259999.97,96
65,None,13,"6,2",Счет-фактура № 660 от 28.08.2019г,Блок- контейнер (улучшенный) разм.5850х2400х2...,шт.,3,9589.95,,28769.85,7.56,217500.07,97
2632,None,61,"38,2",Сч.факт.№ШД000272513/01от 10 июня 2020г,"Труба 400 ЧШГ с ЦПП Ц.54256/6/1,2/7,73",м,40,974.85,,38994,7.68,299473.92,2874
2652,None,65,"40,1",Сч.факт.№ШД000272513/01от 10 июня 2020г,"Манжета Ц.780/1,2/7,73",шт.,11,84.09,,924.99,7.68,7103.92,2894
2710,334,2,4,Сч.фактура № 850от 19.06.2020г,HPL-Лемарк светло-серый -SH 3050х1300х8 FR) (...,л,184,1863.28,,342843.52,7.68,2633038.23,2961
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16485,3471,=D17632+1,3,договорная,Пандус в бокс 230х1200х1800 мм. Металлокаркас ...,компл.,2,39683.336,None,=J17633*I17633,1,=L17633*M17633,17633
16504,3477,14,1,договорная,Сопровождение строительно-монтажных работ со с...,компл.,1,450000,None,450000,1,450000,17665
16505,3478,15,2,договорная,Разработка паспорта благоустройства,компл.,1,400000,None,400000,1,400000,17666
16506,3479,16,3,договорная,Внесение сведений в АИС «Реестр зеленых насажд...,компл.,1,250000,None,250000,1,250000,17667


In [151]:
# Подсчет количества уникальных значений в колонке "Шифр расценки и коды ресурсов"
unique_codes = filtered_df['Шифр расценки и коды ресурсов'].nunique()

# Получение списка уникальных значений
unique_code_list = filtered_df['Шифр расценки и коды ресурсов'].unique()

# Расчёт распределения значений
code_distribution = filtered_df['Шифр расценки и коды ресурсов'].value_counts(normalize=True) * 100

# Вывод результатов
print(f"Количество уникальных значений в колонке 'Шифр расценки и коды ресурсов': {unique_codes}\n")
print("Список уникальных значений:")
for code in unique_code_list:
    print(code)

Количество уникальных значений в колонке 'Шифр расценки и коды ресурсов': 192

Список уникальных значений:
Счет-фактура № 675 от 3 сентября 2019г
Счет-фактура № 660 от 28.08.2019г
Сч.факт.№ШД000272513/01от 10 июня 2020г
Сч.фактура № 850от 19.06.2020г
Сч.фак. №584 от18 июня 2020г
Сч.факт.№848 от 18июня 2020г.
Сч.факт.№851 от 23июня 2020г.
Сч.факт.№50150от 24 июля 2020г
Сч.факт.№774от 29 июня 2020г
Сч.факт.№ 262 от 29 мая 2020г.
Сч.фактура №АН-0008898 от20 июля 2020г
Сч.фактура №АН-0008450 от13 июля 2020г
Сч.фактура №ШД000270242/01 от 25июня 2020г
Сч.фактура №04073/20 от 28 июля 2020г
Сч.факт.№ 262 от 29 мая 2020г
Сч.фактура № 852от 19.06.2020г
Сч.Фак.№ 103 от 23 июня 2020г
Сч.факт.№ 94 от 30.06.2020г
Сч.факт.№637 от16.07.2020
Сч.факт.№ 451 от 22.07.2020
Сч.факт.№466 от 12.08.2020г
Сч.факт.№451 от 22.07.2020г
Сч.факт.№209 от 31.07.2020г.2020г
Сч.факт.№ 057/17508 от 03.08.2020г
Сч.факт.№ 057/17240 от 06.07.2020г
Сч.факт.№ 057/17241от 06.07.2020г
Сч.факт.№ 057/14934от 13.07.2020г
Сч.факт.№

In [90]:
# Сохраняем DataFrame df_filtered_final в Excel-файл
df_filtered_final.to_excel("filtered_data.xlsx", index=False)